# FaceSwap Demo — Google Colab

**使い方:**
1. Cell 3 の `PROJECT_PATH` を自分の Drive パスに変更
2. Cell 5 の `NGROK_TOKEN` を [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken) から取得して入力
3. 上から順にセルを実行
4. Cell 5 に表示された URL をブラウザで開く


In [ ]:
# Cell 1: Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cell 2: 依存パッケージインストール（約1〜2分）
!pip install -q \
    fastapi==0.115.0 \
    "uvicorn[standard]==0.32.0" \
    python-multipart==0.0.12 \
    insightface==0.7.3 \
    onnxruntime-gpu \
    "numpy>=1.24,<2.0" \
    opencv-python-headless==4.10.0.84 \
    Pillow==11.0.0 \
    aiofiles==24.1.0 \
    sse-starlette==2.1.3 \
    pyngrok
print('✅ インストール完了')


In [ ]:
# Cell 3: プロジェクトパス設定 & モデル確認/ダウンロード
import sys
from pathlib import Path

PROJECT_PATH = '/content/drive/MyDrive/faceswap'  # ← Drive 上のパスに変更

sys.path.insert(0, PROJECT_PATH)
print(f'プロジェクト: {PROJECT_PATH}')

model = Path(PROJECT_PATH) / 'models' / 'inswapper_128.onnx'
if not model.exists():
    print('モデルをダウンロード中... (約 554MB、数分かかります)')
    !pip install -q huggingface_hub
    !huggingface-cli download ezioruan/inswapper_128.onnx inswapper_128.onnx \
        --local-dir {PROJECT_PATH}/models
    print('✅ ダウンロード完了・Drive に保存しました')
else:
    print(f'✅ モデル確認済み: {model}')


In [ ]:
# Cell 4: uvicorn をバックグラウンドで起動
import os
import subprocess
import time
import urllib.request

os.chdir(PROJECT_PATH)

proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.main:app',
     '--host', '0.0.0.0', '--port', '8000'],
    env={**os.environ, 'PYTHONPATH': PROJECT_PATH},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print('サーバー起動中... (モデルロードに20〜30秒かかります)')
for _ in range(12):
    time.sleep(5)
    try:
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=3)
        print('✅ サーバー起動完了')
        break
    except Exception:
        print('  ...待機中')
else:
    print('⚠️  起動タイムアウト。ログを確認してください:')
    print(proc.stdout.read(2000).decode(errors='replace'))


In [ ]:
# Cell 5: ngrok トンネル → アプリ URL を表示
from pyngrok import ngrok, conf

NGROK_TOKEN = 'YOUR_NGROK_TOKEN'  # ← https://dashboard.ngrok.com/get-started/your-authtoken

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(8000)

print(f'\n🎉 アプリURL: {tunnel.public_url}')
print('このURLをブラウザで開いてください（スマホも可）')
print(f'デバッグ用: {tunnel.public_url}/?mode=stg')
